In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=160,
    n_features=4,
    n_informative=3,
    n_redundant=0,
    class_sep=1.3,
    random_state=11,
)

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=11
)

baseline = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale")),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)
validation_scores = []

for train_idx, valid_idx in cv.split(X_train, y_train):
    baseline.fit(X_train[train_idx], y_train[train_idx])
    pred = baseline.predict(X_train[valid_idx])
    validation_scores.append(
        balanced_accuracy_score(y_train[valid_idx], pred)
    )

baseline.fit(X_train, y_train)
test_score = balanced_accuracy_score(y_test, baseline.predict(X_test))

In [ ]:
print("Validation scores:", [round(score, 3) for score in validation_scores])
print("Mean validation balanced accuracy: {:.3f}".format(sum(validation_scores) / len(validation_scores)))
print("Test balanced accuracy: {:.3f}".format(test_score))

assert len(validation_scores) == 5
assert all(0.0 <= score <= 1.0 for score in validation_scores)
assert 0.0 <= test_score <= 1.0

In [ ]:
from dataclasses import dataclass

import numpy as np
from sklearn.linear_model import LinearRegression

@dataclass(order=True, frozen=True)
class CircuitCandidate:
    name: str
    n_qubits: int
    depth: int
    two_qubit_gates: int
    parameters: int
    family: str


def summarize_circuit(circuit):
    return [
        circuit.n_qubits,
        circuit.depth,
        circuit.two_qubit_gates,
        circuit.parameters,
        float(circuit.family == "ring"),
        float(circuit.family == "hardware_efficient"),
    ]


def expensive_evaluate(circuit):
    family_penalty = {
        "linear": 0.02,
        "ring": 0.04,
        "hardware_efficient": 0.07,
    }
    return (
        0.04 * circuit.depth
        + 0.09 * circuit.two_qubit_gates
        + 0.015 * circuit.parameters
        + 0.03 * circuit.n_qubits
        + family_penalty[circuit.family]
    )


def make_meta_dataset(records):
    features, targets = zip(*records)
    return np.asarray(features, dtype=float), np.asarray(targets, dtype=float)


experiment_log = []


def update_experiment_log(circuit, verified_target):
    experiment_log.append({
        "circuit": circuit.name,
        "target": verified_target,
    })


initial_candidates = [
    CircuitCandidate("linear-2q-r1", 2, 5, 1, 4, "linear"),
    CircuitCandidate("linear-3q-r1", 3, 8, 2, 6, "linear"),
    CircuitCandidate("ring-3q-r1", 3, 9, 3, 6, "ring"),
    CircuitCandidate("ring-4q-r1", 4, 12, 4, 8, "ring"),
    CircuitCandidate("hea-3q-r2", 3, 14, 6, 12, "hardware_efficient"),
    CircuitCandidate("hea-4q-r2", 4, 18, 8, 16, "hardware_efficient"),
]

new_candidates = [
    CircuitCandidate("linear-4q-r1", 4, 10, 3, 8, "linear"),
    CircuitCandidate("ring-4q-r2", 4, 16, 8, 16, "ring"),
    CircuitCandidate("hea-5q-r1", 5, 15, 6, 10, "hardware_efficient"),
    CircuitCandidate("linear-3q-r2", 3, 11, 4, 9, "linear"),
]

surrogate = LinearRegression()
top_k = 2

In [ ]:
# Each record describes a circuit candidate and an expensive measured target.
# Examples of target: validation loss, transpiled depth, or noisy score.
records = []

for circuit in initial_candidates:
    features = summarize_circuit(circuit)
    target = expensive_evaluate(circuit)
    records.append((features, target))

X_meta, y_meta = make_meta_dataset(records)
surrogate.fit(X_meta, y_meta)

ranked = []
for circuit in new_candidates:
    features = summarize_circuit(circuit)
    predicted_target = surrogate.predict([features])[0]
    ranked.append((predicted_target, circuit))

for _, circuit in sorted(ranked)[:top_k]:
    verified_target = expensive_evaluate(circuit)
    update_experiment_log(circuit, verified_target)

In [ ]:
print("Verified candidates:")
for entry in experiment_log:
    print("{}: {:.3f}".format(entry["circuit"], entry["target"]))

assert len(experiment_log) == top_k
assert all(entry["target"] > 0.0 for entry in experiment_log)